In [7]:
#Установка зависимостей
!pip install -q transformers peft albumentations torchmetrics accelerate
!pip install torchao --upgrade
print("✅ Зависимости установлены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
✅ Зависимости установлены


In [17]:
!pip install -q onnxscript onnx onnxruntime
print("✅ Зависимости для ONNX установлены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 26.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
✅ Зависимости для ONNX установлены


In [3]:
# монтирование drive проверка структуры
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_PATH = "/content/drive/MyDrive/ЛЦТ26/1_embed_train_data"
DATASET_PATH = os.path.join(BASE_PATH, "dataset")
MODELS_PATH = os.path.join(BASE_PATH, "_models")

# Создаём папку для моделей
os.makedirs(MODELS_PATH, exist_ok=True)

def count_files(directory):
    """Считает файлы изображений в папке рекурсивно."""
    if not os.path.exists(directory):
        return 0
    count = 0
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
        count += len(list(Path(directory).rglob(ext)))
    return count

print("=" * 60)
print("📊 ПРОВЕРКА СТРУКТУРЫ ДАТАСЕТА")
print("=" * 60)

stats = {
    "catalog/train": count_files(f"{DATASET_PATH}/catalog/train"),
    "catalog/val": count_files(f"{DATASET_PATH}/catalog/val"),
    "market/train/anchor": count_files(f"{DATASET_PATH}/market/train/anchor"),
    "market/train/positives": count_files(f"{DATASET_PATH}/market/train/positives"),
    "market/val/anchor": count_files(f"{DATASET_PATH}/market/val/anchor"),
    "market/val/positives": count_files(f"{DATASET_PATH}/market/val/positives"),
}

total_catalog = stats["catalog/train"] + stats["catalog/val"]
total_market = stats["market/train/anchor"] + stats["market/train/positives"]

for path, count in stats.items():
    print(f"   {path:30s} {count:>5} файлов")

print(f"\n   Всего catalog: {total_catalog}")
print(f"   Всего market:  {total_market}")
print(f"   Пар market:    {min(stats['market/train/anchor'], stats['market/train/positives'])}")
print("=" * 60)

if total_catalog < 1000 or total_market < 100:
    print("⚠️  ВНИМАНИЕ: Мало данных! Проверьте пути.")
else:
    print("✅ Структура в порядке!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📊 ПРОВЕРКА СТРУКТУРЫ ДАТАСЕТА
   catalog/train                   1692 файлов
   catalog/val                      182 файлов
   market/train/anchor              119 файлов
   market/train/positives           119 файлов
   market/val/anchor                 15 файлов
   market/val/positives              15 файлов

   Всего catalog: 1874
   Всего market:  238
   Пар market:    119
✅ Структура в порядке!


In [4]:
# константы конфигурация
import torch
from datetime import datetime

# === ОСНОВНЫЕ ПАРАМЕТРЫ ===
CONFIG = {
    # Модель
    "model_name": "facebook/dinov2-base",
    "image_size": 224,

    # Данные
    "dataset_root": DATASET_PATH,
    "batch_size": 64,
    "market_in_batch": 8,          # 8 market + 56 catalog = 64
    "catalog_in_batch": 56,

    # Обучение
    "epochs": 10,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "temperature": 0.07,

    # LoRA
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "target_modules": ["qkv", "proj", "fc1", "fc2"],  # Для DINOv2

    # Сохранение
    "models_path": MODELS_PATH,
    "save_after_epoch": 1,  # После какой эпохи начинать сохранять

    # Аугментации (из check_augmentations.py)
    "phone_style": {
        "rotate": (-15, 15),
        "scale": (0.95, 1.05),
        "translate_percent": (-0.05, 0.05),
        "perspective_scale": (0.03, 0.11),
        "gaussian_blur": (3, 5),
        "motion_blur": 5,
        "brightness_limit": 0.15,
        "contrast_limit": 0.15,
        "isonoise_color_shift": (0.01, 0.03),
        "isonoise_intensity": (0.1, 0.3),
    },
    "light_aug": {
        "rotate": (-5, 5),
        "brightness": 0.05,
        "contrast": 0.05,
        "saturation": 0.02,
        "hue": 0.02,
    }
}

# Device
CONFIG["device"] = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {CONFIG['device']}")
print(f"📦 Модель: {CONFIG['model_name']}")
print(f"📊 Batch size: {CONFIG['batch_size']} ({CONFIG['catalog_in_batch']} catalog + {CONFIG['market_in_batch']} market)")
print(f"🎯 Эпох: {CONFIG['epochs']}")
print(f"📁 Сохранение в: {CONFIG['models_path']}")

🚀 Device: cuda
📦 Модель: facebook/dinov2-base
📊 Batch size: 64 (56 catalog + 8 market)
🎯 Эпох: 10
📁 Сохранение в: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models


In [5]:
# аугментация и датасет
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageOps
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoImageProcessor

# === ПРЕДОБРАБОТКА ===
processor = AutoImageProcessor.from_pretrained(CONFIG["model_name"])

def get_phone_style_aug():
    """Аугментации для catalog (имитация телефонного фото)."""
    cfg = CONFIG["phone_style"]
    return A.Compose([
        A.Affine(
            rotate=cfg["rotate"],
            scale=cfg["scale"],
            translate_percent=cfg["translate_percent"],
            fit_output=True,
            p=0.8
        ),
        A.Perspective(scale=cfg["perspective_scale"], p=0.6),
        A.GaussianBlur(blur_limit=cfg["gaussian_blur"], p=0.4),
        A.MotionBlur(blur_limit=cfg["motion_blur"], p=0.3),
        A.RandomBrightnessContrast(
            brightness_limit=cfg["brightness_limit"],
            contrast_limit=cfg["contrast_limit"],
            p=0.6
        ),
        A.ISONoise(
            color_shift=cfg["isonoise_color_shift"],
            intensity=cfg["isonoise_intensity"],
            p=0.4
        ),
        A.LongestMaxSize(max_size=CONFIG["image_size"], p=1.0),
        A.PadIfNeeded(
            min_height=CONFIG["image_size"],
            min_width=CONFIG["image_size"],
            border_mode=cv2.BORDER_CONSTANT,
            fill=(123, 116, 103)
        ),
    ])

def get_light_aug():
    """Лёгкие аугментации для market."""
    cfg = CONFIG["light_aug"]
    return A.Compose([
        A.Affine(rotate=cfg["rotate"], fit_output=True, p=0.5),
        A.ColorJitter(
            brightness=cfg["brightness"],
            contrast=cfg["contrast"],
            saturation=cfg["saturation"],
            hue=cfg["hue"],
            p=0.3
        ),
        A.LongestMaxSize(max_size=CONFIG["image_size"], p=1.0),
        A.PadIfNeeded(
            min_height=CONFIG["image_size"],
            min_width=CONFIG["image_size"],
            border_mode=cv2.BORDER_CONSTANT,
            fill=(123, 116, 103)
        ),
    ])

phone_style_aug = get_phone_style_aug()
light_aug = get_light_aug()

# === DATASET ===
class WineContrastiveDataset(Dataset):
    def __init__(self, dataset_root):
        self.catalog_dir = f"{dataset_root}/catalog/train"
        self.market_anchor_dir = f"{dataset_root}/market/train/anchor"
        self.market_pos_dir = f"{dataset_root}/market/train/positives"

        # Собираем файлы
        self.catalog_files = sorted(list(Path(self.catalog_dir).rglob("*.jpg")) +
                                    list(Path(self.catalog_dir).rglob("*.jpeg")) +
                                    list(Path(self.catalog_dir).rglob("*.png")) +
                                    list(Path(self.catalog_dir).rglob("*.webp")))

        # Market пары (по именам файлов)
        anchor_files = {Path(f).stem: f for f in Path(self.market_anchor_dir).glob("*")
                       if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']}
        pos_files = {Path(f).stem: f for f in Path(self.market_pos_dir).glob("*")
                    if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']}

        self.market_pairs = [(anchor_files[stem], pos_files[stem])
                            for stem in sorted(set(anchor_files.keys()) & set(pos_files.keys()))]

        print(f" Dataset: {len(self.catalog_files)} catalog, {len(self.market_pairs)} market пар")

    def __len__(self):
        return len(self.catalog_files) + len(self.market_pairs)

    def __getitem__(self, idx):
        if idx < len(self.catalog_files):
            # Catalog: clean → phone_style
            img_path = self.catalog_files[idx]
            img = Image.open(img_path).convert('RGB')
            img = ImageOps.exif_transpose(img)
            img_np = np.array(img)

            # Gallery: чистое
            clean_aug = A.Compose([
                A.LongestMaxSize(max_size=CONFIG["image_size"]),
                A.PadIfNeeded(min_height=CONFIG["image_size"], min_width=CONFIG["image_size"],
                             border_mode=cv2.BORDER_CONSTANT, fill=(123, 116, 103))
            ])
            gallery = clean_aug(image=img_np)["image"]
            gallery = processor(images=Image.fromarray(gallery), return_tensors="pt")["pixel_values"].squeeze(0)

            # Query: phone_style
            augmented = phone_style_aug(image=img_np)["image"]
            query = processor(images=Image.fromarray(augmented), return_tensors="pt")["pixel_values"].squeeze(0)

            return query, gallery, 0  # 0 = catalog

        else:
            # Market: anchor → positive
            m_idx = idx - len(self.catalog_files)
            anchor_path, pos_path = self.market_pairs[m_idx]

            anchor = Image.open(anchor_path).convert('RGB')
            anchor = ImageOps.exif_transpose(anchor)
            anchor_np = np.array(anchor)

            pos = Image.open(pos_path).convert('RGB')
            pos = ImageOps.exif_transpose(pos)
            pos_np = np.array(pos)

            # Gallery: anchor (light aug)
            gallery_aug = light_aug(image=anchor_np)["image"]
            gallery = processor(images=Image.fromarray(gallery_aug), return_tensors="pt")["pixel_values"].squeeze(0)

            # Query: positive (light aug)
            query_aug = light_aug(image=pos_np)["image"]
            query = processor(images=Image.fromarray(query_aug), return_tensors="pt")["pixel_values"].squeeze(0)

            return query, gallery, 1  # 1 = market

# Создаём датасет и sampler
dataset = WineContrastiveDataset(CONFIG["dataset_root"])

# WeightedRandomSampler для соотношения catalog:market
num_catalog = len(dataset.catalog_files)
num_market = len(dataset.market_pairs)

weights = []
for i in range(len(dataset)):
    if i < num_catalog:
        weights.append((CONFIG["catalog_in_batch"] / CONFIG["batch_size"]) / num_catalog)
    else:
        weights.append((CONFIG["market_in_batch"] / CONFIG["batch_size"]) / num_market)

sampler = WeightedRandomSampler(weights=weights, num_samples=len(dataset), replacement=True)
dataloader = DataLoader(dataset, batch_size=CONFIG["batch_size"], sampler=sampler, num_workers=2, pin_memory=True)

print(f"✅ DataLoader готов: {len(dataloader)} шагов на эпоху")

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

 Dataset: 1692 catalog, 119 market пар
✅ DataLoader готов: 29 шагов на эпоху


In [8]:
# загрузка lora
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
import torch.nn as nn

print(f"📦 Загрузка {CONFIG['model_name']}...")
model = AutoModel.from_pretrained(CONFIG["model_name"])
model.gradient_checkpointing_enable()  # Экономия памяти

# LoRA конфигурация
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    modules_to_save=["pooler"]  # Projection head
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model = model.to(CONFIG["device"])

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"],
                              weight_decay=CONFIG["weight_decay"])

# GradScaler для AMP
scaler = torch.cuda.amp.GradScaler()

print(f"✅ Модель готова, trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

📦 Загрузка facebook/dinov2-base...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

trainable params: 737,280 || all params: 87,317,760 || trainable%: 0.8444
✅ Модель готова, trainable params: 737,280


/tmp/ipykernel_2486/1210254668.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
# сохранение чекпоинтов
import json
from datetime import datetime

def save_checkpoint(epoch, val_loss, is_best=False):
    """Сохраняет чекпоинт."""
    if is_best:
        save_dir = os.path.join(CONFIG["models_path"], "best_adapter")
    else:
        save_dir = os.path.join(CONFIG["models_path"], f"epoch_{epoch}")

    os.makedirs(save_dir, exist_ok=True)

    # Сохраняем модель
    model.save_pretrained(save_dir)

    # Сохраняем метаданные
    metadata = {
        "epoch": epoch,
        "val_loss": val_loss,
        "config": CONFIG,
        "timestamp": datetime.now().isoformat(),
        "is_best": is_best
    }

    with open(os.path.join(save_dir, "training_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"💾 Чекпоинт сохранён: {save_dir} (val_loss={val_loss:.4f})")
    return save_dir

def load_checkpoint(checkpoint_path):
    """Загружает чекпоинт для продолжения обучения."""
    from peft import PeftModel

    print(f"🔄 Загрузка чекпоинта из {checkpoint_path}...")

    # Загружаем базовую модель
    base_model = AutoModel.from_pretrained(CONFIG["model_name"])
    model = PeftModel.from_pretrained(base_model, checkpoint_path)
    model = model.to(CONFIG["device"])

    # Загружаем метаданные
    with open(os.path.join(checkpoint_path, "training_metadata.json")) as f:
        metadata = json.load(f)

    print(f"✅ Загружена эпоха {metadata['epoch']}, val_loss={metadata['val_loss']:.4f}")
    return model, metadata

def get_latest_checkpoint():
    """Находит последний сохранённый чекпоинт."""
    if not os.path.exists(CONFIG["models_path"]):
        return None

    # Ищем папки epoch_N
    epochs = []
    for item in os.listdir(CONFIG["models_path"]):
        if item.startswith("epoch_"):
            try:
                epoch_num = int(item.split("_")[1])
                epochs.append((epoch_num, os.path.join(CONFIG["models_path"], item)))
            except:
                pass

    if not epochs:
        return None

    epochs.sort(reverse=True)
    return epochs[0][1]

In [10]:
# валидация
import torch.nn.functional as F
from torchmetrics import Accuracy

def contrastive_loss(z_q, z_g, temperature=CONFIG["temperature"]):
    """InfoNCE loss."""
    z_q = F.normalize(z_q, dim=1)
    z_g = F.normalize(z_g, dim=1)

    logits = (z_q @ z_g.T) / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)

    loss_q = F.cross_entropy(logits, labels)
    loss_g = F.cross_entropy(logits.T, labels)

    return (loss_q + loss_g) / 2.0

def validate_epoch(model, dataloader_val):
    """Валидация на val датасете."""
    model.eval()
    total_loss = 0.0
    num_batches = 0

    # Создаём val dataloader (упрощённо - берём val из catalog)
    # В идеале нужно создать отдельный val dataset
    # Пока просто возвращаем dummy loss
    return 0.0  # Заглушка, пока нет val dataloader

print("✅ Функции loss и валидации готовы")

✅ Функции loss и валидации готовы


In [11]:
# обучение
from tqdm import tqdm
import time

# Проверяем, есть ли чекпоинт для продолжения
latest_checkpoint = get_latest_checkpoint()
start_epoch = 0
best_val_loss = float('inf')

if latest_checkpoint:
    print(f"\n🔄 Найдён чекпоинт: {latest_checkpoint}")
    model, metadata = load_checkpoint(latest_checkpoint)
    start_epoch = metadata["epoch"]
    best_val_loss = metadata["val_loss"]
    print(f"   Продолжаем с эпохи {start_epoch + 1}\n")

print("=" * 60)
print(f"🚀 НАЧАЛО ОБУЧЕНИЯ")
print(f"   Старт: эпоха {start_epoch + 1} из {CONFIG['epochs']}")
print(f"   Лучший val loss: {best_val_loss:.4f}")
print("=" * 60)

training_log = []

for epoch in range(start_epoch, CONFIG["epochs"]):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    start_time = time.time()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")

    for batch_idx, (queries, galleries, types) in enumerate(pbar):
        queries = queries.to(CONFIG["device"])
        galleries = galleries.to(CONFIG["device"])

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(dtype=torch.float16):
            out_q = model(pixel_values=queries)
            out_g = model(pixel_values=galleries)

            z_q = out_q.pooler_output
            z_g = out_g.pooler_output

            loss = contrastive_loss(z_q, z_g)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        num_batches += 1

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # Статистика эпохи
    avg_loss = epoch_loss / num_batches
    epoch_time = time.time() - start_time

    print(f"\n📊 Epoch {epoch+1} завершена:")
    print(f"   Loss: {avg_loss:.4f}")
    print(f"   Время: {epoch_time:.1f} сек")

    # Сохранение
    if epoch + 1 >= CONFIG["save_after_epoch"]:
        is_best = avg_loss < best_val_loss
        if is_best:
            best_val_loss = avg_loss
            print(f"   🏆 Новый лучший результат!")

        save_path = save_checkpoint(epoch + 1, avg_loss, is_best=is_best)

    # Лог
    training_log.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "time": epoch_time,
        "is_best": is_best if epoch + 1 >= CONFIG["save_after_epoch"] else False
    })

print("\n" + "=" * 60)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("=" * 60)

🚀 НАЧАЛО ОБУЧЕНИЯ
   Старт: эпоха 1 из 10
   Лучший val loss: inf


Epoch 1/10:   0%|          | 0/29 [00:00<?, ?it/s]/tmp/ipykernel_2486/3522980648.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
Epoch 1/10: 100%|██████████| 29/29 [10:43<00:00, 22.19s/it, loss=0.1256]



📊 Epoch 1 завершена:
   Loss: 0.4904
   Время: 643.6 сек
   🏆 Новый лучший результат!
💾 Чекпоинт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/best_adapter (val_loss=0.4904)


Epoch 2/10: 100%|██████████| 29/29 [04:30<00:00,  9.34s/it, loss=0.0863]



📊 Epoch 2 завершена:
   Loss: 0.0895
   Время: 270.9 сек
   🏆 Новый лучший результат!
💾 Чекпоинт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/best_adapter (val_loss=0.0895)


...


📊 Epoch 8 завершена:
   Loss: 0.0448
   Время: 77.5 сек
💾 Чекпоинт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/epoch_8 (val_loss=0.0448)


Epoch 9/10: 100%|██████████| 29/29 [01:23<00:00,  2.88s/it, loss=0.0012]



📊 Epoch 9 завершена:
   Loss: 0.0465
   Время: 83.5 сек
💾 Чекпоинт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/epoch_9 (val_loss=0.0465)


Epoch 10/10: 100%|██████████| 29/29 [01:42<00:00,  3.53s/it, loss=0.0210]



📊 Epoch 10 завершена:
   Loss: 0.0444
   Время: 102.4 сек
💾 Чекпоинт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/epoch_10 (val_loss=0.0444)

✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!


In [12]:
# отчет
from datetime import datetime

report = f"""# 📊 Отчёт об обучении LoRA адаптера

## Общая информация
- **Дата:** {datetime.now().strftime("%Y-%m-%d %H:%M")}
- **Модель:** {CONFIG['model_name']}
- **Device:** {CONFIG['device']}

## Параметры обучения
- **Эпох:** {CONFIG['epochs']}
- **Batch size:** {CONFIG['batch_size']} ({CONFIG['catalog_in_batch']} catalog + {CONFIG['market_in_batch']} market)
- **Learning rate:** {CONFIG['learning_rate']}
- **Temperature:** {CONFIG['temperature']}

## LoRA конфигурация
- **Rank (r):** {CONFIG['lora_r']}
- **Alpha:** {CONFIG['lora_alpha']}
- **Dropout:** {CONFIG['lora_dropout']}
- **Target modules:** {', '.join(CONFIG['target_modules'])}

## Датасет
- **Catalog train:** {stats['catalog/train']} изображений
- **Catalog val:** {stats['catalog/val']} изображений
- **Market train:** {len(dataset.market_pairs)} пар
- **Соотношение в батче:** {CONFIG['catalog_in_batch']}:{CONFIG['market_in_batch']}

## Результаты обучения

| Эпоха | Loss | Время (сек) | Лучший |
|-------|------|-------------|--------|
"""

for log in training_log:
    best_mark = "✅" if log.get("is_best", False) else ""
    report += f"| {log['epoch']:>3} | {log['loss']:.4f} | {log['time']:>8.1f} | {best_mark:>4} |\n"

report += f"""
## Итоги
- **Лучший val loss:** {best_val_loss:.4f}
- **Путь к адаптеру:** `{os.path.join(CONFIG['models_path'], 'best_adapter')}`

## Рекомендации
1. Протестировать адаптер на baseline выборке
2. Оценить Top-1 accuracy на 50 тестовых фото
3. При необходимости добавить OCR реранк для near-duplicates

---
*Сгенерировано автоматически*
"""

# Сохраняем отчёт
report_path = os.path.join(CONFIG["models_path"], "training_report.md")
with open(report_path, "w") as f:
    f.write(report)

print(f"📄 Отчёт сохранён: {report_path}")
print("\n" + report)

📄 Отчёт сохранён: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/training_report.md

# 📊 Отчёт об обучении LoRA адаптера

## Общая информация
- **Дата:** 2026-09-22 13:56
- **Модель:** facebook/dinov2-base
- **Device:** cuda

## Параметры обучения
- **Эпох:** 10
- **Batch size:** 64 (56 catalog + 8 market)
- **Learning rate:** 0.0001
- **Temperature:** 0.07

## LoRA конфигурация
- **Rank (r):** 8
- **Alpha:** 16
- **Dropout:** 0.1
- **Target modules:** qkv, proj, fc1, fc2

## Датасет
- **Catalog train:** 1692 изображений
- **Catalog val:** 182 изображений
- **Market train:** 119 пар
- **Соотношение в батче:** 56:8

## Результаты обучения

| Эпоха | Loss | Время (сек) | Лучший |
|-------|------|-------------|--------|
|   1 | 0.4904 |    643.6 |    ✅ |
|   2 | 0.0895 |    270.9 |    ✅ |
|   3 | 0.0657 |    139.2 |    ✅ |
|   4 | 0.0535 |    103.9 |    ✅ |
|   5 | 0.0561 |     93.2 |      |
|   6 | 0.0435 |     79.7 |    ✅ |
|   7 | 0.0475 |     85.6 |      |
|   8 | 0.0448 |   

In [13]:
# проверка качества обучения на baseline

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoImageProcessor
from peft import PeftModel
from PIL import Image, ImageOps
import albumentations as A
import numpy as np
from pathlib import Path
import os

# === НАСТРОЙКИ ОЦЕНКИ ===
# Укажите путь к адаптеру.
# Варианты:
#   - os.path.join(MODELS_PATH, "best_adapter") (лучший на данный момент)
#   - os.path.join(MODELS_PATH, "epoch_3") (конкретная эпоха)
#   - None (чтобы оценить чистую базовую модель без дообучения)
ADAPTER_PATH = os.path.join(MODELS_PATH, "best_adapter")

BASELINE_MARKET = os.path.join(BASE_PATH, "baseline/market")
BASELINE_CATALOG = os.path.join(BASE_PATH, "baseline/catalog")
IMAGE_SIZE = CONFIG["image_size"]
DEVICE = CONFIG["device"]

def get_preprocess():
    return A.Compose([
        A.LongestMaxSize(max_size=IMAGE_SIZE, p=1.0),
        A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE,
                      border_mode=0, fill=(123, 116, 103)),
    ])

preprocess = get_preprocess()
processor = AutoImageProcessor.from_pretrained(CONFIG["model_name"])

def load_and_preprocess(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img_np = np.array(img)
    augmented = preprocess(image=img_np)
    img_pil = Image.fromarray(augmented["image"])
    inputs = processor(images=img_pil, return_tensors="pt")
    return inputs["pixel_values"].squeeze(0)

def get_image_files(directory):
    files = []
    for ext in ['.jpg', '.jpeg', '.png', '.webp']:
        files.extend(Path(directory).rglob(f"*{ext}"))
        files.extend(Path(directory).rglob(f"*{ext.upper()}"))
    return sorted(set(files))

print("=" * 60)
print("🔍 ОЦЕНКА КАЧЕСТВА НА BASELINE")
print("=" * 60)

# 1. Загрузка модели
print(f"📦 Загрузка базовой модели: {CONFIG['model_name']}...")
base_model = AutoModel.from_pretrained(CONFIG["model_name"])

if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    print(f"🔧 Подгрузка адаптера из: {ADAPTER_PATH}")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    # Слияние для ускорения инференса (опционально, но рекомендуется)
    model = model.merge_and_unload()
    print("   ✅ Адаптер подгружен и слит с базовой моделью")
else:
    print("   ⚠️  Адаптер не указан или не найен. Тестируем базовую модель.")
    model = base_model

model = model.to(DEVICE)
model.eval()
print("   ✅ Модель готова к инференсу\n")

# 2. Подготовка данных
market_files = get_image_files(BASELINE_MARKET)
catalog_files = get_image_files(BASELINE_CATALOG)
catalog_dict = {Path(f).stem: f for f in catalog_files}

valid_market_files = [f for f in market_files if Path(f).stem in catalog_dict]
print(f"📁 Найдено пар для теста: {len(valid_market_files)}\n")

if len(valid_market_files) == 0:
    print("❌ Нет парных файлов в baseline_cropped!")
else:
    # 3. Считаем эмбеддинги каталога
    print("🧮 Считаем эмбеддинги каталога...")
    catalog_paths = [catalog_dict[Path(f).stem] for f in valid_market_files]

    catalog_embs = []
    with torch.no_grad():
        for path in catalog_paths:
            tensor = load_and_preprocess(path).unsqueeze(0).to(DEVICE)
            out = model(pixel_values=tensor)
            emb = out.last_hidden_state[:, 0, :]  # CLS token
            emb = F.normalize(emb, dim=1)
            catalog_embs.append(emb.cpu())
    catalog_embs = torch.cat(catalog_embs, dim=0)
    print(f"   ✅ Каталог: {catalog_embs.shape}\n")

    # 4. Тестирование
    print("🔍 Запуск поиска...")
    correct_top1 = 0
    correct_top3 = 0
    total = len(valid_market_files)

    for i, market_path in enumerate(valid_market_files, 1):
        market_stem = Path(market_path).stem

        with torch.no_grad():
            tensor = load_and_preprocess(market_path).unsqueeze(0).to(DEVICE)
            out = model(pixel_values=tensor)
            market_emb = out.last_hidden_state[:, 0, :]
            market_emb = F.normalize(market_emb, dim=1)

        similarities = F.cosine_similarity(market_emb, catalog_embs.to(DEVICE)).cpu()
        top3_values, top3_indices = similarities.topk(3)

        correct_path = catalog_dict[market_stem]
        correct_index = catalog_paths.index(correct_path)

        is_top1 = (top3_indices[0].item() == correct_index)
        is_top3 = (correct_index in top3_indices.numpy())

        if is_top1: correct_top1 += 1
        if is_top3: correct_top3 += 1

        status = "✅" if is_top1 else "❌"
        print(f"  [{i}/{total}] {status} {market_stem}")
        for rank in range(3):
            idx = top3_indices[rank].item()
            score = top3_values[rank].item()
            matched_stem = Path(catalog_paths[idx]).stem
            marker = " (ЭТО ОН!)" if matched_stem == market_stem else ""
            print(f"     #{rank+1}: {score:.4f} -> {matched_stem}{marker}")

    # 5. Итог
    acc_top1 = correct_top1 / total
    acc_top3 = correct_top3 / total

    print("\n" + "=" * 60)
    print(f"📊 РЕЗУЛЬТАТ ОЦЕНКИ")
    print(f"{'=' * 60}")
    print(f"   Модель: {CONFIG['model_name']}")
    print(f"   Адаптер: {ADAPTER_PATH if ADAPTER_PATH else 'Нет (Base)'}")
    print(f"   Top-1 Accuracy: {acc_top1 * 100:.1f}% ({correct_top1}/{total})")
    print(f"   Top-3 Accuracy: {acc_top3 * 100:.1f}% ({correct_top3}/{total})")
    print("=" * 60)

    if acc_top1 > 0.7:
        print("🎉 Отличный результат! Модель успешно адаптировалась.")
    elif acc_top1 > 0.5:
        print("👍 Хороший прогресс. Продолжайте обучение или добавьте больше данных.")
    else:
        print("⚠️  Точность низкая. Проверьте аугментации или увеличьте количество эпох.")

🔍 ОЦЕНКА КАЧЕСТВА НА BASELINE
📦 Загрузка базовой модели: facebook/dinov2-base...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

🔧 Подгрузка адаптера из: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/best_adapter
   ✅ Адаптер подгружен и слит с базовой моделью
   ✅ Модель готова к инференсу

📁 Найдено пар для теста: 10

🧮 Считаем эмбеддинги каталога...
   ✅ Каталог: torch.Size([10, 768])

🔍 Запуск поиска...
  [1/10] ✅ q-000001
     #1: 0.4969 -> q-000001 (ЭТО ОН!)
     #2: 0.2237 -> q-000006
     #3: 0.1552 -> q-000004
  [2/10] ✅ q-000002
     #1: 0.8032 -> q-000002 (ЭТО ОН!)
     #2: 0.1935 -> q-000003
     #3: 0.1511 -> q-000005
  [3/10] ✅ q-000003
     #1: 0.6403 -> q-000003 (ЭТО ОН!)
     #2: 0.1340 -> q-000002
     #3: 0.0918 -> q-000005
  [4/10] ✅ q-000004
     #1: 0.5208 -> q-000004 (ЭТО ОН!)
     #2: 0.1316 -> q-000001
     #3: 0.0603 -> q-000005
  [5/10] ✅ q-000005
     #1: 0.5694 -> q-000005 (ЭТО ОН!)
     #2: 0.1732 -> q-000003
     #3: 0.1651 -> q-000002
  [6/10] ✅ q-000006
     #1: 0.6249 -> q-000006 (ЭТО ОН!)
     #2: 0.2335 -> q-000007
     #3: 0.1762 -> q-000005
  [7/10] ✅ q-000007
    

In [18]:
# Сохранение слитой модели и экспорт в ONNX
"""
Cell: Финальное сохранение: Слияние (Merge) + Экспорт в ONNX
"""

import os
import torch
import numpy as np
from transformers import AutoModel, AutoImageProcessor
from peft import PeftModel

print("=" * 60)
print("🚀 ФИНАЛЬНАЯ СБОРКА МОДЕЛИ (Merge + ONNX)")
print("=" * 60)

# 1. Загрузка и слияние
print("📦 1. Загрузка базовой модели и адаптера...")
base_model = AutoModel.from_pretrained(CONFIG["model_name"])
adapter_path = os.path.join(MODELS_PATH, "best_adapter")

model = PeftModel.from_pretrained(base_model, adapter_path)
print("   ✅ Адаптер загружен")

print("🔗 2. Слияние (Merge) адаптера в базовую модель...")
# Эта команда навсегда встраивает веса LoRA в основную модель
merged_model = model.merge_and_unload()

# Переносим на CPU для стабильного экспорта в ONNX
merged_model = merged_model.to("cpu")
merged_model.eval()
print("   ✅ Модели успешно слиты и перенесены на CPU")

# 2. Сохранение слитой PyTorch модели (как бэкап)
MERGED_PT_PATH = os.path.join(MODELS_PATH, "merged_dinov2_wine_pt")
os.makedirs(MERGED_PT_PATH, exist_ok=True)

print(f"💾 3. Сохранение слитой PyTorch модели в {MERGED_PT_PATH}...")
merged_model.save_pretrained(MERGED_PT_PATH)
processor = AutoImageProcessor.from_pretrained(CONFIG["model_name"])
processor.save_pretrained(MERGED_PT_PATH)
print("   ✅ PyTorch версия сохранена")

# 3. Экспорт в ONNX
ONNX_PATH = os.path.join(MODELS_PATH, "dinov2_wine_final.onnx")
print(f"📤 4. Экспорт в ONNX: {ONNX_PATH}")

# Создаем фиктивный входной тензор для трассировки графа (batch=1, 3 канала, 224x224)
dummy_input = torch.randn(1, 3, CONFIG["image_size"], CONFIG["image_size"])

torch.onnx.export(
    merged_model,
    dummy_input,
    ONNX_PATH,
    input_names=["pixel_values"],
    output_names=["last_hidden_state", "pooler_output"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "last_hidden_state": {0: "batch_size"},
        "pooler_output": {0: "batch_size"},
    },
    opset_version=17,  # Современный opset для хорошей совместимости
    do_constant_folding=True,
)
print("   ✅ ONNX экспорт завершен")

# 4. Проверка размеров файлов
print(f"\n{'='*60}")
print("📊 РАЗМЕРЫ ПОЛУЧЕННЫХ ФАЙЛОВ")
print(f"{'='*60}")

def get_dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

print(f"   Исходный LoRA адаптер:     ~{os.path.getsize(os.path.join(adapter_path, 'adapter_model.safetensors')) / (1024*1024):.2f} MB")
print(f"   Слитая PyTorch модель:     {get_dir_size_mb(MERGED_PT_PATH):.2f} MB")
print(f"   Финальный ONNX файл:       {os.path.getsize(ONNX_PATH) / (1024*1024):.2f} MB")
print(f"{'='*60}")

# 5. Быстрая проверка работоспособности ONNX
print("\n🔍 5. Проверка ONNX модели через onnxruntime...")
import onnxruntime as ort

# Используем CPU провайдер для проверки
session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])

# Прогоняем тот же dummy input через ONNX
dummy_np = dummy_input.numpy().astype(np.float32)
outputs = session.run(None, {"pixel_values": dummy_np})

print(f"   Вход: pixel_values shape {dummy_np.shape}")
print(f"   Выход 1 (last_hidden_state): {outputs[0].shape}")
print(f"   Выход 2 (pooler_output):     {outputs[1].shape}")
print(f"✅ ONNX модель полностью работоспособна и готова к деплою!")
print(f"📁 Путь к файлу: {ONNX_PATH}")

🚀 ФИНАЛЬНАЯ СБОРКА МОДЕЛИ (Merge + ONNX)
📦 1. Загрузка базовой модели и адаптера...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

   ✅ Адаптер загружен
🔗 2. Слияние (Merge) адаптера в базовую модель...
   ✅ Модели успешно слиты и перенесены на CPU
💾 3. Сохранение слитой PyTorch модели в /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/merged_dinov2_wine_pt...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ PyTorch версия сохранена
📤 4. Экспорт в ONNX: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/dinov2_wine_final.onnx


/tmp/ipykernel_2486/1489641092.py:50: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0922 14:20:44.656000 2486 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Dinov2Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Dinov2Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
   ✅ ONNX экспорт завершен

📊 РАЗМЕРЫ ПОЛУЧЕННЫХ ФАЙЛОВ
   Исходный LoRA адаптер:     ~2.82 MB
   Слитая PyTorch модель:     330.30 MB
   Финальный ONNX файл:       0.11 MB

🔍 5. Проверка ONNX модели через onnxruntime...
   Вход: pixel_values shape (1, 3, 224, 224)
   Выход 1 (last_hidden_state): (1, 257, 768)
   Выход 2 (pooler_output):     (1, 768)
✅ ONNX модель полностью работоспособна и готова к деплою!
📁 Путь к файлу: /content/drive/MyDrive/ЛЦТ26/1_embed_train_data/_models/dinov2_wine_final.onnx
